# Blackjack CV — Entrenamiento YOLOv8 en Google Colab

Entrena el modelo de detección de cartas. Ejecútalo en Google Colab
para aprovechar la GPU gratuita (T4 es suficiente).

## Pasos previos (en tu Raspberry Pi)

### Opción A — Fotos reales *(recomendado para mayor precisión)*

**1.** Captura ≥100 fotos por clase (14 clases = ≥1400 fotos en total):
```bash
python scripts/capture_dataset.py
```

**2.** Auto-genera las anotaciones YOLO:
```bash
python scripts/auto_annotate.py --preview   # revisa las bboxes visualmente
python scripts/auto_annotate.py             # o directo sin preview
```

**3.** Comprime y sube a Google Drive:
```bash
cd data && zip -r labeled.zip labeled/
```
Sube `labeled.zip` a cualquier carpeta de tu Drive.

---

### Opción B — Datos sintéticos *(arranque rápido, menor precisión)*

**1.** Genera el dataset sintético:
```bash
python scripts/generate_synthetic_data.py --n 3000 --out data/synthetic
```

**2.** Comprime y sube:
```bash
cd data && zip -r synthetic.zip synthetic/
```
Sube `synthetic.zip` a Google Drive.

---

Después ejecuta las celdas de este notebook en orden.

## 1. Verificar entorno y GPU

In [ ]:
import os

if not os.path.exists('/content'):
    raise RuntimeError('Ejecuta este notebook en Google Colab')

import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU disponible:', result.stdout.strip())
else:
    print('⚠️  Sin GPU — ve a Entorno de ejecución → Cambiar tipo → T4 GPU')

## 2. Instalar dependencias

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
print('Ultralytics instalado correctamente')

## 3. Montar Google Drive y extraer dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import glob

# Busca primero fotos reales (labeled.zip), luego datos sintéticos (synthetic.zip)
ZIP_PATH = None
for name in ('labeled.zip', 'synthetic.zip'):
    matches = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    if matches:
        ZIP_PATH = matches[0]
        print(f'Encontrado ({name}): {ZIP_PATH}')
        break

if ZIP_PATH is None:
    raise FileNotFoundError(
        'No se encontró labeled.zip ni synthetic.zip en Google Drive.\n'
        'Consulta las instrucciones en la primera celda.'
    )

In [ ]:
import zipfile, pathlib

DATASET_DIR = pathlib.Path('/content/dataset')
DATASET_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATASET_DIR)

# Localizar dataset.yaml
yamls = list(DATASET_DIR.rglob('dataset.yaml'))
if not yamls:
    raise FileNotFoundError('dataset.yaml no encontrado dentro del zip')

YAML_PATH = str(yamls[0])

# Corregir la ruta 'path' dentro del yaml para que apunte a Colab
import re
content = pathlib.Path(YAML_PATH).read_text()
content = re.sub(r'^path:.*$', f'path: {yamls[0].parent}', content, flags=re.MULTILINE)
pathlib.Path(YAML_PATH).write_text(content)

print(f'Dataset extraído en: {DATASET_DIR}')
print(f'dataset.yaml: {YAML_PATH}')

# Contar imágenes
n_train = len(list((yamls[0].parent / 'images' / 'train').glob('*.jpg')))
n_val   = len(list((yamls[0].parent / 'images' / 'val').glob('*.jpg')))
print(f'Imágenes — train: {n_train}  val: {n_val}')

## 4. Entrenar el modelo

In [ ]:
# Parámetros de entrenamiento
# Con fotos reales: epochs=100, patience=20 da mejores resultados
# Con datos sintéticos: epochs=50 es suficiente
EPOCHS    = 100    # aumenta si el modelo no converge (monitoriza mAP50 en logs)
IMG_SIZE  = 640
BATCH     = 16     # reduce a 8 si hay error de memoria
MODEL     = 'yolov8n.pt'   # nano: rápido y ligero, suficiente para la Pi

model = YOLO(MODEL)

results = model.train(
    data     = YAML_PATH,
    epochs   = EPOCHS,
    imgsz    = IMG_SIZE,
    batch    = BATCH,
    name     = 'blackjack_v1',
    project  = '/content/runs',
    patience = 20,        # para si no mejora en 20 épocas
    # Augmentación — reduce agresividad para fotos reales con poco dato
    degrees  = 15,        # rotación máxima
    translate= 0.1,
    scale    = 0.4,
    fliplr   = 0.5,
    mosaic   = 0.5,       # baja a 0.0 si hay sobreajuste
    verbose  = True,
)

print('\nEntrenamiento completado.')
print(f'mAP50: {results.results_dict.get("metrics/mAP50(B)", "N/A")}')
print('Objetivo con fotos reales: mAP50 > 0.90')

## 5. Evaluar el modelo

In [ ]:
best_model_path = '/content/runs/blackjack_v1/weights/best.pt'
best = YOLO(best_model_path)
metrics = best.val(data=YAML_PATH)

print(f'mAP50   : {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')
print(f'Precision: {metrics.box.p.mean():.3f}')
print(f'Recall  : {metrics.box.r.mean():.3f}')

## 6. Ver ejemplos de detección

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, random

val_images = glob.glob(str(yamls[0].parent / 'images' / 'val' / '*.jpg'))
sample     = random.sample(val_images, min(4, len(val_images)))

results_pred = best(sample, verbose=False)

fig, axes = plt.subplots(1, len(sample), figsize=(16, 5))
if len(sample) == 1:
    axes = [axes]

for ax, r in zip(axes, results_pred):
    ax.imshow(r.plot()[:, :, ::-1])
    ax.axis('off')

plt.suptitle('Detecciones en imágenes de validación')
plt.tight_layout()
plt.show()

## 7. Guardar el modelo en Google Drive

In [ ]:
import shutil, pathlib

DRIVE_DEST = pathlib.Path('/content/drive/MyDrive/blackjack-cv/models')
DRIVE_DEST.mkdir(parents=True, exist_ok=True)

dest = DRIVE_DEST / 'yolov8n_blackjack.pt'
shutil.copy(best_model_path, dest)

print(f'Modelo guardado en Drive: {dest}')

## 8. Copiar el modelo a la Raspberry Pi

Descarga el archivo `yolov8n_blackjack.pt` desde Google Drive y cópialo a tu Pi:

**Opción A — desde el navegador:**
1. Abre Google Drive → carpeta `blackjack-cv/models/`
2. Descarga `yolov8n_blackjack.pt`
3. Cópialo a la Pi con un USB o con `scp`

**Opción B — directamente en la Pi con gdown:**
```bash
pip install gdown
gdown 'URL_DEL_ARCHIVO' -O ~/projects/blackjack-cv/models/yolov8n_blackjack.pt
```
(La URL la obtienes haciendo clic derecho → Compartir → Copiar enlace en Drive)

Una vez el modelo esté en `models/yolov8n_blackjack.pt`, arranca el sistema:
```bash
cd ~/projects/blackjack-cv
python main.py
```